# Extreme Temperature Indices

**Standards:**
- Baseline: 1995-2014 (IPCC AR6)
- Resolution: ERA5 LAND 0.1° (~11 km)

In [1]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import geemap
import ee, datetime
import numpy as np

In [2]:
# Access Earth Engine
ee.Authenticate()
ee.Initialize(
    project = 'eecc-maureen',
    opt_url = 'https://earthengine-highvolume.googleapis.com'
)

In [3]:
now_iso = datetime.datetime.utcnow().isoformat() + "Z"

In [5]:
def with_common_metadata(img, units, metric, roi_name, source, extra=None):
    props = {
        "creator": "Earth Engine",
        "exported_at": now_iso,
        "units": units,                   
        "metric": metric,              
        "roi_name": roi_name,
        "source": source
    }
    if extra:
        props.update(extra)
    return img.set(props)

In [6]:
def plot(index, roi, scale=1000, palette=['lightblue','blue','purple']):
    # Get band name
    band = index.bandNames().getInfo()[0]

    # Compute stats over ROI for auto visualization
    stats = index.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=roi,
        scale=scale,
        maxPixels=1e13
    ).getInfo()

    # Use explicit keys instead of list(stats.values())
    vmin = stats.get(f"{band}_min")
    vmax = stats.get(f"{band}_max")

    # Safety check — if masked, choose default or raise clear message
    if (vmin is None) or (vmax is None):
        print("⚠️ No valid data inside ROI at this scale — trying coarser resolution (5000m)")
        return plot(index, roi, scale=5000, palette=palette)

    vis = {'min': vmin, 'max': vmax, 'palette': palette}

    # Retrieve metric name safely (metadata may not exist)
    props = index.getInfo().get("properties", {})
    metric = props.get("metric", band)

    # Build map
    Map = geemap.Map()
    Map.centerObject(roi, 10)
    Map.addLayer(index, vis, f"{metric} [{vmin:.1f}-{vmax:.1f}]")
    Map.add_colorbar(vis_params=vis, label=f"{metric} (mm)")
    return Map

In [7]:
def K_to_C(img):
    return img.subtract(273.15)

# Region

In [8]:
# Place name
place_name = "Porto Alegre, Brazil"
#place_name = "Brazil"

# Get place boundary
roi = ox.geocode_to_gdf(place_name)
roi = gpd.GeoDataFrame(roi, geometry='geometry', crs='EPSG:4326')

# Convert geometry to Earth Engine format - Handle both Polygon and MultiPolygon
geom = roi.geometry.iloc[0]

if geom.geom_type == 'Polygon':
    coords = [list(geom.exterior.coords)]
    roi_ee = ee.Geometry.Polygon(coords)
elif geom.geom_type == 'MultiPolygon':
    polygons = []
    for poly in geom.geoms:
        polygons.append(list(poly.exterior.coords))
    roi_ee = ee.Geometry.MultiPolygon(polygons)
else:
    raise ValueError(f"Unexpected geometry type: {geom.geom_type}")

print(f"Geometry type: {geom.geom_type}")
print(f"Earth Engine geometry created: {roi_ee.getInfo()['type']}")

Geometry type: MultiPolygon
Earth Engine geometry created: MultiPolygon


# Collections and Constants

In [9]:
start_date, end_date = '2024-01-01', '2025-01-01'
 
year = 2024

BASE_START, BASE_END = '1995-01-01', '2015-01-01'
BASE_START_YEAR, BASE_END_YEAR = 1995, 2014

In [10]:
tmax_base = (
    ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
    .select('temperature_2m_max')
    .filterDate(BASE_START, BASE_END)
    .filterBounds(roi_ee)
)

tmin_base = (
    ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
    .select('temperature_2m_min')
    .filterDate(BASE_START, BASE_END)
    .filterBounds(roi_ee)
    .select('temperature_2m_min')
)

In [11]:
tmax = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .select("temperature_2m_max")
    .filterDate(start_date, end_date)
    .filterBounds(roi_ee)
)

tmin = (
    ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
    .select('temperature_2m_min')
    .filterDate(start_date, end_date)
    .filterBounds(roi_ee)
)

# TXx

### Annual Calculations

In [ ]:
def txx_for_year(ic_daily, year, roi=None, band_name="temperature_2m_max"):
    """
    TXx for a given year:
    - ic_daily: ImageCollection with daily Tmax (ERA5-Land DAILY_AGGR)
    - year: integer (e.g. 2024)
    """
    year = int(year)
    start = ee.Date.fromYMD(year, 1, 1)
    end   = start.advance(1, "year")

    year_ic = (
        ic_daily
        .filterDate(start, end)
        .select(band_name)
    )

    # Max over the year
    txx = year_ic.max().rename("TXx_K")

    # Convert to °C
    txx = txx.subtract(273.15).rename("TXx")


    if roi is not None:
        txx = txx.clip(roi)

    return txx

In [ ]:
def txx_climatology(ic, base_start, base_end, roi=None, band_name="temperature_2m_max"):
    """
    TXx climatology:
    - Compute TXx for each year in the baseline
    - Take the mean of these annual TXx layers
    Returns a single-band ee.Image in °C.
    """
    
    years = ee.List.sequence(int(base_start), int(base_end))

    def per_year(y):
        y = ee.Number(y)
        start = ee.Date.fromYMD(y, 1, 1)
        end   = start.advance(1, "year")

        # Annual TXx
        txx = ic.filterDate(start, end).select(band_name).max().rename("TXx")

        if roi:
            txx = txx.clip(roi)

        # only metadata AFTER band exists
        txx = txx.set({
            "year": y,
            "metric": "TXx"
        })
        return txx

    # ImageCollection of yearly TXx (each with 1 band)
    annual_ic = ee.ImageCollection(years.map(per_year))

    # mean of all TXx maps
    clim = annual_ic.mean().rename(f"TXx_clim_{base_start}_{base_end}_C")

    # Convert to °C
    clim = clim.subtract(273.15).rename("TXx_clim")

    if roi:
        clim = clim.clip(roi)

    return clim

In [ ]:
# TXx for baseline climatology (1995–2014)
txx_clim = txx_climatology(tmax_base, BASE_START_YEAR, BASE_END_YEAR, roi=roi_ee)

# TXx for 2024
txx = txx_for_year(tmax, 2024, roi=roi_ee)

In [ ]:
txx = with_common_metadata(
    txx,
    units = "C",                                  # Temperature units
    metric = "TXx",                             # annual max temperature
    # roi_name = "Porto Alegre, Brazil",
    roi_name = "Brazil",
    source = "ECMWF/ERA5_LAND/DAILY_AGGR",
    extra = {
        "type": "single_year",                    # distinguishes from climatology
        "year": 2024,
        "description": "Annual maximum Tmax for the year 2024"
    }
)


txx_clim = with_common_metadata(
    txx_clim,
    units = "C",
    metric = "TXx",
    # roi_name = "Porto Alegre, Brazil",
    roi_name = "Brazil",
    source = "ECMWF/ERA5_LAND/DAILY_AGGR",
    extra = {
        "type": "climatology",
        "baseline_start_year": 1995,
        "baseline_end_year": 2014,
        "statistic": "mean",                      
        "description": (
            "Climatological mean of annual maximum Tmax "
            "computed from TXx across 1995–2014 baseline."
        )
    }
)

In [ ]:
txx_clim

In [ ]:
# values = txx.reduceRegion(
#     reducer=ee.Reducer.toList(),
#     geometry=roi_ee,
#     scale=1000,
#     maxPixels=1e13
# ).getInfo()

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=txx_clim,
    description="TXx_Climatology",
    folder="GEE_exports",
    fileNamePrefix="TXx_climatology",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

task = ee.batch.Export.image.toDrive(
    image=txx,
    description="TXx_2024",
    folder="GEE_exports",
    fileNamePrefix="TXx_2024",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

### Monthly Calculations

In [21]:
def txx_for_month(ic_daily, year, month, roi=None, band_name="temperature_2m_max"):
    """
    TXx for a single month.
    - ic_daily: ImageCollection with daily Tmax (ERA5-Land DAILY_AGGR)
    - year: integer (e.g. 2024)
    - month: integer (1-12)
    Returns maximum temperature for that month in °C.
    """
    # Build start/end dates (server- or client-side)
    if isinstance(year, (int, float)) and isinstance(month, (int, float)):
        # Client-side
        start = f"{int(year)}-{int(month):02d}-01"
        if month == 12:
            end = f"{int(year)+1}-01-01"
        else:
            end = f"{int(year)}-{int(month)+1:02d}-01"
    else:
        # Server-side
        y = ee.Number(year).int()
        m = ee.Number(month).int()
        start = ee.Date.fromYMD(y, m, 1)
        end = start.advance(1, 'month')
    
    month_ic = (
        ic_daily
        .filterDate(start, end)
        .select(band_name)
    )
    
    # Max over the month
    txx = month_ic.max().rename("TXx_K")
    
    # Convert to °C
    txx = txx.subtract(273.15).rename("TXx").set('year', year).set('month', month)
    
    if roi is not None:
        txx = txx.clip(roi)
    
    return txx

def txx_monthly_climatology(ic, base_start_year, base_end_year, month, roi=None,
                           band_name="temperature_2m_max",
                           reducer=ee.Reducer.mean()):
    """
    TXx climatology for a specific month across baseline years.
    `month` can be int (1-12) or ee.Number.
    Returns mean/median TXx for that month across the baseline period.
    """
    years = ee.List.sequence(int(base_start_year), int(base_end_year))
    
    monthly = ee.ImageCollection(
        years.map(lambda y: txx_for_month(ic, y, month, roi=roi, band_name=band_name))
    )
    
    clim = monthly.reduce(reducer).rename(f"TXx_clim_month{int(month)}_{int(base_start_year)}_{int(base_end_year)}")
    if roi is not None:
        clim = clim.clip(roi)
    return clim

In [22]:
# Loop through all 12 months to compute TXx climatology and 2024 values
txx_monthly_clim = {}
txx_monthly_2024 = {}

for month in range(1, 13):
    print(f"Computing TXx for month {month}...")
    
    # Climatology (mean TXx for this month across 1995-2014)
    txx_monthly_clim[month] = txx_monthly_climatology(
        tmax_base, BASE_START_YEAR, BASE_END_YEAR, month=month, roi=roi_ee
    )
    
    # TXx for this month in 2024
    txx_monthly_2024[month] = txx_for_month(
        tmax, 2024, month=month, roi=roi_ee
    )

print("✓ Completed computing TXx for all 12 months")

# Access results like:
# txx_monthly_clim[1]  # January climatology
# txx_monthly_2024[1]  # January 2024
# etc.

Computing TXx for month 1...
Computing TXx for month 2...
Computing TXx for month 3...
Computing TXx for month 4...
Computing TXx for month 5...
Computing TXx for month 6...
Computing TXx for month 7...
Computing TXx for month 8...
Computing TXx for month 9...
Computing TXx for month 10...
Computing TXx for month 11...
Computing TXx for month 12...
✓ Completed computing TXx for all 12 months


In [25]:
# Month names for file naming
month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
               7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}

# Add metadata and export TXx monthly indices
for month in range(1, 13):
    month_name = month_names[month]
    
    # Add metadata to climatology
    txx_monthly_clim[month] = with_common_metadata(
        txx_monthly_clim[month],
        units="°C",
        metric="TXx",
        roi_name="Porto Alegre, Brazil",
        source="ECMWF/ERA5_LAND/DAILY_AGGR",
        extra={
            "type": "climatology",
            "baseline_start_year": 1995,
            "baseline_end_year": 2014,
            "statistic": "mean",
            "month": month,
            "description": (
                f"Climatological mean of monthly maximum temperature "
                f"for {month_name} computed from TXx across 1995–2014 baseline."
            )
        }
    )
    
    # Add metadata to 2024 value
    txx_monthly_2024[month] = with_common_metadata(
        txx_monthly_2024[month],
        units="°C",
        metric="TXx",
        roi_name="Porto Alegre, Brazil",
        source="ECMWF/ERA5_LAND/DAILY_AGGR",
        extra={
            "type": "single_month",
            "year": 2024,
            "month": month,
            "description": f"Monthly maximum temperature for {month_name} 2024"
        }
    )
    
    # Export climatology
    task = ee.batch.Export.image.toDrive(
        image=txx_monthly_clim[month],
        description=f"TXx_Climatology_{month_name}",
        folder="GEE_exports",
        fileNamePrefix=f"TXx_climatology_{month_name}",
        region=roi_ee,
        scale=11132,  # ERA5 LAND native resolution
        crs="EPSG:4326",
        maxPixels=1e13
    )
    task.start()
    
    # Export 2024 value
    task = ee.batch.Export.image.toDrive(
        image=txx_monthly_2024[month],
        description=f"TXx_2024_{month_name}",
        folder="GEE_exports",
        fileNamePrefix=f"TXx_2024_{month_name}",
        region=roi_ee,
        scale=11132,
        crs="EPSG:4326",
        maxPixels=1e13
    )
    task.start()
    
    print(f"✓ Exported TXx for {month_name}")

print("\n✓ Completed exporting all TXx monthly indices")


✓ Exported TXx for Jan
✓ Exported TXx for Feb
✓ Exported TXx for Mar
✓ Exported TXx for Apr
✓ Exported TXx for May
✓ Exported TXx for Jun
✓ Exported TXx for Jul
✓ Exported TXx for Aug
✓ Exported TXx for Sep
✓ Exported TXx for Oct
✓ Exported TXx for Nov
✓ Exported TXx for Dec

✓ Completed exporting all TXx monthly indices


# TNx

### Annual Calculations

In [ ]:
def tnx_for_year(ic, year, roi=None, band_name="temperature_2m_min"):
    """Annual maximum of daily Tmin (°C)"""

    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    annual_max_K = (
        ic.filterDate(start, end)
          .select(band_name)
          .max()                      # still in Kelvin
          .rename(f"TNx_{year}_K")
          .set("year", year)
    )

    if roi:
        annual_max_K = annual_max_K.clip(roi)

    # Convert AFTER aggregation
    annual_max_C = annual_max_K.subtract(273.15).rename(f"TNx_{year}")

    return annual_max_C

def tnx_climatology(ic, base_start, base_end, roi=None, band_name="temperature_2m_min"):
    """
    Climatology of TNx over baseline period (mean of annual maxima, °C).
    Assumes:
      - ic: daily Tmin in Kelvin (e.g. ERA5-Land DAILY_AGGR)
    """

    base_start = int(base_start)
    base_end = int(base_end)
    years = ee.List.sequence(base_start, base_end)

    def compute_year_tnx(y):
        y = ee.Number(y).int()
        start = ee.Date.fromYMD(y, 1, 1)
        end = start.advance(1, "year")

        # Annual max in Kelvin
        annual_max_K = (
            ic.filterDate(start, end)
              .select(band_name)
              .max()
              .rename("TNx_K")       # same band name for every year
              .set("year", y)
        )

        if roi:
            annual_max_K = annual_max_K.clip(roi)

        # Convert to °C
        annual_max_C = (
            annual_max_K
            .subtract(273.15)
            .rename("TNx_C")        # same band name for every year
            .set("year", y)
        )

        return annual_max_C

    # Collection of annual TNx images (in °C, 1 band: 'TNx_C')
    annual_ic_C = ee.ImageCollection(years.map(compute_year_tnx))

    # Mean over years → climatology in °C
    clim = (
        annual_ic_C
        .mean()
        .rename(f"TNx_clim_{base_start}_{base_end}_C")
        .set({
            "baseline_start_year": base_start,
            "baseline_end_year": base_end,
            "metric": "TNx"
        })
    )

    if roi:
        clim = clim.clip(roi)

    return clim

In [ ]:
# TNx for baseline climatology (1995–2014)
tnx_clim = tnx_climatology(tmin_base, BASE_START_YEAR, BASE_END_YEAR, roi=roi_ee)
# TNx for 2024
tnx = tnx_for_year(tmin, 2024, roi=roi_ee)

In [ ]:
tnx = with_common_metadata(
    tnx,
    units = "C",                                  # Temperature units
    metric = "TNx",                               # annual max tmin
    # roi_name = "Porto Alegre, Brazil",
    roi_name = "Brazil",
    source = "ECMWF/ERA5_LAND/DAILY_AGGR",
    extra = {
        "type": "single_year",                    # distinguishes from climatology
        "year": 2024,
        "description": "Annual maximum Tmin for the year 2024"
    }
)


tnx_clim = with_common_metadata(
    tnx_clim,
    units = "C",
    metric = "TNx",
    # roi_name = "Porto Alegre, Brazil",
    roi_name = "Brazil",
    source = "ECMWF/ERA5_LAND/DAILY_AGGR",
    extra = {
        "type": "climatology",
        "baseline_start_year": 1995,
        "baseline_end_year": 2014,
        "statistic": "mean",                      
        "description": (
            "Climatological mean of annual maximum Tmin "
            "computed from TNx across 1995–2014 baseline."
        )
    }
)

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=tnx_clim,
    description="TNx_Climatology",
    folder="GEE_exports",
    fileNamePrefix="TNx_climatology",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

task = ee.batch.Export.image.toDrive(
    image=tnx,
    description="TNx_2024",
    folder="GEE_exports",
    fileNamePrefix="TNx_2024",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

### Monthly Calculations

In [30]:
def tnx_for_month(ic, year, month, roi=None, band_name="temperature_2m_min"):
    """
    Monthly maximum of daily Tmin (°C) for a single month.
    - ic: ImageCollection with daily Tmin
    - year: integer (e.g. 2024) or ee.Number
    - month: integer (1-12) or ee.Number
    """
    # Build start/end dates (server- or client-side)
    if isinstance(year, (int, float)) and isinstance(month, (int, float)):
        start = f"{int(year)}-{int(month):02d}-01"
        if month == 12:
            end = f"{int(year)+1}-01-01"
        else:
            end = f"{int(year)}-{int(month)+1:02d}-01"
    else:
        y = ee.Number(year).int()
        m = ee.Number(month).int()
        start = ee.Date.fromYMD(y, m, 1)
        end = start.advance(1, 'month')
    
    monthly_max_K = (
        ic.filterDate(start, end)
          .select(band_name)
          .max()                      # still in Kelvin
          .rename("TNx_K")            # Use generic name instead of f-string
    )
    
    # Set properties (ee objects are fine here)
    monthly_max_K = monthly_max_K.set("year", year).set("month", month)
    
    if roi:
        monthly_max_K = monthly_max_K.clip(roi)
    
    # Convert AFTER aggregation
    monthly_max_C = monthly_max_K.subtract(273.15).rename("TNx")
    
    return monthly_max_C

def tnx_monthly_climatology(ic, base_start_year, base_end_year, month, roi=None,
                           band_name="temperature_2m_min",
                           reducer=ee.Reducer.mean()):
    """
    TNx climatology for a specific month across baseline years.
    `month` can be int (1-12) or ee.Number.
    Returns mean/median TNx for that month across the baseline period.
    """
    month_ee = ee.Number(int(month))
    month_int = int(month)
    
    years = ee.List.sequence(int(base_start_year), int(base_end_year))
    
    monthly = ee.ImageCollection(
        years.map(lambda y: tnx_for_month(ic, y, month_ee, roi=roi, band_name=band_name))
    )
    
    clim = monthly.reduce(reducer).rename(f"TNx_clim_month{month_int}_{int(base_start_year)}_{int(base_end_year)}")
    if roi is not None:
        clim = clim.clip(roi)
    return clim

In [31]:
# Loop through all 12 months to compute TNx climatology and 2024 values
tnx_monthly_clim = {}
tnx_monthly_2024 = {}

for month in range(1, 13):
    print(f"Computing TNx for month {month}...")
    
    # Climatology (mean TNx for this month across 1995-2014)
    tnx_monthly_clim[month] = tnx_monthly_climatology(
        tmin_base, BASE_START_YEAR, BASE_END_YEAR, month=month, roi=roi_ee
    )
    
    # TNx for this month in 2024
    tnx_monthly_2024[month] = tnx_for_month(
        tmin, 2024, month=month, roi=roi_ee
    )

print("✓ Completed computing TNx for all 12 months")

# Access results like:
# tnx_monthly_clim[1]  # January climatology
# tnx_monthly_2024[1]  # January 2024
# etc.

Computing TNx for month 1...
Computing TNx for month 2...
Computing TNx for month 3...
Computing TNx for month 4...
Computing TNx for month 5...
Computing TNx for month 6...
Computing TNx for month 7...
Computing TNx for month 8...
Computing TNx for month 9...
Computing TNx for month 10...
Computing TNx for month 11...
Computing TNx for month 12...
✓ Completed computing TNx for all 12 months


In [32]:
# Add metadata and export TNx monthly indices
for month in range(1, 13):
    month_name = month_names[month]
    
    # Add metadata to climatology
    tnx_monthly_clim[month] = with_common_metadata(
        tnx_monthly_clim[month],
        units="°C",
        metric="TNx",
        roi_name="Porto Alegre, Brazil",
        source="ECMWF/ERA5_LAND/DAILY_AGGR",
        extra={
            "type": "climatology",
            "baseline_start_year": 1995,
            "baseline_end_year": 2014,
            "statistic": "mean",
            "month": month,
            "description": (
                f"Climatological mean of monthly maximum minimum temperature "
                f"for {month_name} computed from TNx across 1995–2014 baseline."
            )
        }
    )
    
    # Add metadata to 2024 value
    tnx_monthly_2024[month] = with_common_metadata(
        tnx_monthly_2024[month],
        units="°C",
        metric="TNx",
        roi_name="Porto Alegre, Brazil",
        source="ECMWF/ERA5_LAND/DAILY_AGGR",
        extra={
            "type": "single_month",
            "year": 2024,
            "month": month,
            "description": f"Monthly maximum minimum temperature for {month_name} 2024"
        }
    )
    
    # Export climatology
    task = ee.batch.Export.image.toDrive(
        image=tnx_monthly_clim[month],
        description=f"TNx_Climatology_{month_name}",
        folder="GEE_exports",
        fileNamePrefix=f"TNx_climatology_{month_name}",
        region=roi_ee,
        scale=11132,
        crs="EPSG:4326",
        maxPixels=1e13
    )
    task.start()
    
    # Export 2024 value
    task = ee.batch.Export.image.toDrive(
        image=tnx_monthly_2024[month],
        description=f"TNx_2024_{month_name}",
        folder="GEE_exports",
        fileNamePrefix=f"TNx_2024_{month_name}",
        region=roi_ee,
        scale=11132,
        crs="EPSG:4326",
        maxPixels=1e13
    )
    task.start()
    
    print(f"✓ Exported TNx for {month_name}")

print("\n✓ Completed exporting all TNx monthly indices")

✓ Exported TNx for Jan
✓ Exported TNx for Feb
✓ Exported TNx for Mar
✓ Exported TNx for Apr
✓ Exported TNx for May
✓ Exported TNx for Jun
✓ Exported TNx for Jul
✓ Exported TNx for Aug
✓ Exported TNx for Sep
✓ Exported TNx for Oct
✓ Exported TNx for Nov
✓ Exported TNx for Dec

✓ Completed exporting all TNx monthly indices


# TXxxp

## Annual Values

In [ ]:
def txx_threshold_annual(ic_base, percentile=90, band_name="temperature_2m_max"):
    def k_to_c(img):
        return (img
                .select(band_name)
                .subtract(273.15)
                .copyProperties(img, img.propertyNames()))

    ic_C = ic_base.map(k_to_c)

    thr_C = (
        ic_C
        .reduce(ee.Reducer.percentile([percentile]))
        .toFloat()
        .rename(f"thr_p{percentile}_C")
    )
    return thr_C

In [ ]:
def txxp_for_year(ic, year, thr_img, band_name="temperature_2m_max",
                  roi=None, percentile=90):
    """
    TXxp for a single year:
    - ic: daily Tmax ImageCollection in Kelvin (ERA5-Land DAILY_AGGR)
    - thr_img: baseline threshold in °C (from txx_threshold_annual)
    Returns an ee.Image with two bands (both Float32):
      - 'TX{p}p_pct'  : % of days above threshold
      - 'TX{p}p_days' : # of days above threshold
    """

    # Build date range
    if isinstance(year, (int, float)):
        start = f"{int(year)}-01-01"
        end = f"{int(year)+1}-01-01"
        y_prop = int(year)
    else:
        y = ee.Number(year).int()
        start = ee.Date.fromYMD(y, 1, 1)
        end = start.advance(1, "year")
        y_prop = y

    # Kelvin -> °C
    def k_to_c(img):
        return (img
                .select(band_name)
                .subtract(273.15)
                .copyProperties(img, img.propertyNames()))

    year_ic_C = ic.filterDate(start, end).map(k_to_c)

    # Mark days above threshold
    def mark_hot(img):
        return img.gt(thr_img).selfMask()

    hot_days = year_ic_C.map(mark_hot)

    # Count hot days (UInt32 by default)
    hot_count = hot_days.count()

    # Cast to Float32
    hot_count_f = hot_count.toFloat()

    total_days = ee.Number(year_ic_C.size())

    # Percent (% of days)
    txxp_pct = hot_count_f.divide(total_days).multiply(100.0)

    # Build final image, both bands float
    txxp_img = (
        txxp_pct.rename(f"TX{percentile}p_pct")
        .addBands(hot_count_f.rename(f"TX{percentile}p_days"))
        .toFloat()   # ensure both are Float32
        .set({
            "year": y_prop,
            "metric": f"TX{percentile}p",
            "threshold_percentile": percentile
        })
    )

    if roi:
        txxp_img = txxp_img.clip(roi)

    return txxp_img



In [ ]:
def txxp_climatology(ic, base_start_year, base_end_year, thr_img,
                     band_name="temperature_2m_max",
                     roi=None, percentile=90,
                     reducer=ee.Reducer.mean()):
    """
    TXxp climatology (mean or median across years), using a fixed baseline threshold.
    Returns an ee.Image with two Float32 bands:
      - 'TX{p}p_clim_{start}_{end}_pct'
      - 'TX{p}p_clim_{start}_{end}_days'
    """

    base_start_year = int(base_start_year)
    base_end_year = int(base_end_year)
    years = ee.List.sequence(base_start_year, base_end_year)

    def per_year(y):
        return txxp_for_year(ic, y, thr_img,
                             band_name=band_name,
                             roi=roi,
                             percentile=percentile)

    annual = ee.ImageCollection(years.map(per_year))

    pct_band_name = f"TX{percentile}p_pct"
    days_band_name = f"TX{percentile}p_days"

    annual_pct = annual.select(pct_band_name)
    annual_days = annual.select(days_band_name)

    # These reduces can become Float64 → cast back to Float32
    pct_clim = (
        annual_pct
        .reduce(reducer)
        .toFloat()
        .rename(f"TX{percentile}p_clim_{base_start_year}_{base_end_year}_pct")
    )

    days_clim = (
        annual_days
        .reduce(reducer)
        .toFloat()
        .rename(f"TX{percentile}p_clim_{base_start_year}_{base_end_year}_days")
    )

    clim = (
        pct_clim
        .addBands(days_clim)
        .toFloat()
        .set({
            "baseline_start_year": base_start_year,
            "baseline_end_year": base_end_year,
            "metric": f"TX{percentile}p"
        })
    )

    if roi:
        clim = clim.clip(roi)

    return clim


### TX90p

In [ ]:
# Baseline thresholds
tx90_thr = txx_threshold_annual(tmax_base, percentile=90)

# Annual indices (2024)
tx90p = txxp_for_year(tmax, 2024, tx90_thr, roi=roi_ee, percentile=90)

# Climatologies (1995–2014)
tx90p_clim = txxp_climatology(tmax_base, BASE_START_YEAR, BASE_END_YEAR,
                              tx90_thr, roi=roi_ee, percentile=90)

In [ ]:
tx90p = with_common_metadata(
    tx90p,
    units = "str(%) or days",                                  # percentage or days
    metric = "TX90p",                             # percentage of days when Tmax > 90th percentile of baseline Tmax
    # roi_name = "Porto Alegre, Brazil",
    roi_name = "Brazil",
    source = "ECMWF/ERA5_LAND/DAILY_AGGR",
    extra = {
        "type": "single_year",                    # distinguishes from climatology
        "year": 2024,
        "description": "Percentage of days when Tmax > 90th percentile of baseline Tmax for the year 2024"
    }
)


tx90p_clim = with_common_metadata(
    tx90p_clim,
    units = "str(%) or days",
    metric = "TX90p",
    # roi_name = "Porto Alegre, Brazil",
    roi_name = "Brazil",
    source = "ECMWF/ERA5_LAND/DAILY_AGGR",
    extra = {
        "type": "climatology",
        "baseline_start_year": 1995,
        "baseline_end_year": 2014,
        "statistic": "mean",                      
        "description": (
            "Climatological mean of percentage of days when Tmax > 90th percentile of baseline Tmax "
            "computed from TX90p across 1995–2014 baseline."
        )
    }
)

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=tx90p_clim,
    description="TX90p_Climatology",
    folder="GEE_exports",
    fileNamePrefix="TX90p_climatology",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

task = ee.batch.Export.image.toDrive(
    image=tx90p,
    description="TX90p_2024",
    folder="GEE_exports",
    fileNamePrefix="TX90p_2024",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

### TX99p

In [ ]:
# Baseline thresholds
tx99_thr = txx_threshold_annual(tmax_base, percentile=99)

# Annual indices (2024)
tx99p = txxp_for_year(tmax, 2024, tx99_thr, roi=roi_ee, percentile=99)

# Climatologies (1995–2014)
tx99p_clim = txxp_climatology(tmax_base, BASE_START_YEAR, BASE_END_YEAR,
                              tx99_thr, roi=roi_ee, percentile=99)

In [ ]:
tx99p = with_common_metadata(
    tx99p,
    units = "str(%) or days",                                  # percentage or days
    metric = "TX99p",                             # percentage of days when Tmax > 99th percentile of baseline Tmax
    # roi_name = "Porto Alegre, Brazil",
    roi_name = "Brazil",
    source = "ECMWF/ERA5_LAND/DAILY_AGGR",
    extra = {
        "type": "single_year",                    # distinguishes from climatology
        "year": 2024,
        "description": "Percentage of days when Tmax > 99th percentile of baseline Tmax for the year 2024"
    }
)


tx99p_clim = with_common_metadata(
    tx99p_clim,
    units = "str(%) or days",
    metric = "TX99p",
    # roi_name = "Porto Alegre, Brazil",
    roi_name = "Brazil",
    source = "ECMWF/ERA5_LAND/DAILY_AGGR",
    extra = {
        "type": "climatology",
        "baseline_start_year": 1995,
        "baseline_end_year": 2014,
        "statistic": "mean",                      
        "description": (
            "Climatological mean of percentage of days when Tmax > 99th percentile of baseline Tmax "
            "computed from TX99p across 1995–2014 baseline."
        )
    }
)

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=tx99p_clim,
    description="TX99p_Climatology",
    folder="GEE_exports",
    fileNamePrefix="TX99p_climatology",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

task = ee.batch.Export.image.toDrive(
    image=tx99p,
    description="TX99p_2024",
    folder="GEE_exports",
    fileNamePrefix="TX99p_2024",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

## Monthly Calculations 

In [33]:
# ---------- 1) Monthly threshold from baseline ----------
def txx_threshold_monthly(ic_base, month, percentile=90, band_name="temperature_2m_max"):
    """
    Compute a monthly baseline percentile threshold image.
    `month` can be int (1-12) or ee.Number.
    Returns an ee.Image with threshold in °C.
    """
    # Convert month to ee.Number for consistent handling
    if isinstance(month, (int, float)):
        m = ee.Number(int(month))
    else:
        m = ee.Number(month).int()
    
    # Filter to the specific month across all baseline years
    def filter_month(img):
        month_num = ee.Date(img.get('system:time_start')).get('month')
        return img.updateMask(month_num.eq(m))
    
    # Convert Kelvin -> °C
    def k_to_c(img):
        return (img
                .select(band_name)
                .subtract(273.15)
                .copyProperties(img, img.propertyNames()))
    
    # Filter to month and convert to °C
    base_monthly = (ic_base
                    .map(filter_month)
                    .map(k_to_c))
    
    # Get month as int for band naming
    month_int = int(month) if isinstance(month, (int, float)) else month.getInfo()
    thr = (base_monthly.reduce(ee.Reducer.percentile([percentile]))
                  .toFloat()
                  .rename(f"thr_p{percentile}_month{month_int}_C"))
    return thr


# ---------- 2) TXxxp for a single month ----------
def txxp_for_month(ic, year, month, thr_img, band_name="temperature_2m_max",
                   roi=None, percentile=90):
    """
    TXxp for a single month:
    - ic: daily Tmax ImageCollection in Kelvin (ERA5-Land DAILY_AGGR)
    - thr_img: baseline threshold in °C (from txx_threshold_monthly)
    Returns an ee.Image with two bands:
      - 'TX{p}p_pct'  : % of days above threshold
      - 'TX{p}p_days' : # of days above threshold
    """
    # Build month date range (server- or client-side safe)
    if isinstance(year, (int, float)) and isinstance(month, (int, float)):
        start = f"{int(year)}-{int(month):02d}-01"
        if month == 12:
            end = f"{int(year)+1}-01-01"
        else:
            end = f"{int(year)}-{int(month)+1:02d}-01"
        y_prop = int(year)
        m_prop = int(month)
    else:
        y = ee.Number(year).int()
        m = ee.Number(month).int()
        start = ee.Date.fromYMD(y, m, 1)
        end = start.advance(1, 'month')
        y_prop = year
        m_prop = month
    
    # Kelvin -> °C
    def k_to_c(img):
        return (img
                .select(band_name)
                .subtract(273.15)
                .copyProperties(img, img.propertyNames()))
    
    month_ic_C = ic.filterDate(start, end).map(k_to_c)
    
    # Mark days above threshold
    def mark_hot(img):
        return img.gt(thr_img).selfMask()
    
    hot_days = month_ic_C.map(mark_hot)
    
    # Count hot days
    hot_count = hot_days.count()
    hot_count_f = hot_count.toFloat()
    
    total_days = ee.Number(month_ic_C.size())
    
    # Percent (% of days)
    txxp_pct = hot_count_f.divide(total_days).multiply(100.0)
    
    # Build final image, both bands float
    txxp_img = (
        txxp_pct.rename(f"TX{percentile}p_pct")
        .addBands(hot_count_f.rename(f"TX{percentile}p_days"))
        .toFloat()
        .set({
            "year": y_prop,
            "month": m_prop,
            "metric": f"TX{percentile}p",
            "threshold_percentile": percentile
        })
    )
    
    if roi:
        txxp_img = txxp_img.clip(roi)
    
    return txxp_img


# ---------- 3) TXxxp monthly climatology across baseline ----------
def txxp_monthly_climatology(ic, base_start_year, base_end_year, month, thr_img,
                             band_name="temperature_2m_max",
                             roi=None, percentile=90, reducer=ee.Reducer.mean()):
    """
    Compute TXxxp climatology for a specific month (mean/median across years) using a fixed baseline threshold.
    `month` can be int (1-12) or ee.Number.
    Returns a single ee.Image with two bands: TXxxp_pct, TXxxp_days (climatological mean/median).
    """
    years = ee.List.sequence(int(base_start_year), int(base_end_year))
    
    monthly = ee.ImageCollection(
        years.map(lambda y: txxp_for_month(ic, y, month, thr_img,
                                          band_name=band_name,
                                          roi=roi,
                                          percentile=percentile))
    )
    
    # Reduce across years → mean/median of each band
    clim_raw = monthly.reduce(reducer)
    
    # Band names become e.g. 'TX90p_pct_mean', 'TX90p_days_mean'. Normalize to clean names:
    band_pct = [b for b in clim_raw.bandNames().getInfo() if f'TX{percentile}p_pct' in b][0]
    band_days = [b for b in clim_raw.bandNames().getInfo() if f'TX{percentile}p_days' in b][0]
    
    clim = (clim_raw.select([band_pct, band_days], [f"TX{percentile}p_pct", f"TX{percentile}p_days"])
                  .toFloat()
                  .set('baseline_start_year', int(base_start_year))
                  .set('baseline_end_year', int(base_end_year))
                  .set('month', int(month)))
    
    if roi is not None:
        clim = clim.clip(roi)
    
    return clim

In [34]:
# Loop through all 12 months to compute TX90p climatology and 2024 values
# Step 1: Compute monthly thresholds for all months
tx90_thr_monthly = {}
print("Computing monthly thresholds for TX90p...")
for month in range(1, 13):
    print(f"  Computing threshold for month {month}...")
    tx90_thr_monthly[month] = txx_threshold_monthly(
        tmax_base, month=month, percentile=90, band_name='temperature_2m_max'
    )
print("✓ Completed computing thresholds for all 12 months\n")

# Step 2: Compute TX90p climatology and 2024 values for all months
tx90p_monthly_clim = {}
tx90p_monthly_2024 = {}

for month in range(1, 13):
    print(f"Computing TX90p for month {month}...")
    
    # Climatology (mean TX90p for this month across 1995-2014)
    tx90p_monthly_clim[month] = txxp_monthly_climatology(
        tmax_base, BASE_START_YEAR, BASE_END_YEAR, month=month,
        thr_img=tx90_thr_monthly[month], band_name='temperature_2m_max', roi=roi_ee,
        percentile=90, reducer=ee.Reducer.mean()
    )
    
    # TX90p for this month in 2024
    tx90p_monthly_2024[month] = txxp_for_month(
        tmax, 2024, month=month, thr_img=tx90_thr_monthly[month],
        band_name='temperature_2m_max', roi=roi_ee, percentile=90
    )

print("✓ Completed computing TX90p for all 12 months")

# Access results like:
# tx90p_monthly_clim[1]  # January climatology (has bands: TX90p_pct, TX90p_days)
# tx90p_monthly_2024[1]  # January 2024 (has bands: TX90p_pct, TX90p_days)
# etc.

Computing monthly thresholds for TX90p...
  Computing threshold for month 1...
  Computing threshold for month 2...
  Computing threshold for month 3...
  Computing threshold for month 4...
  Computing threshold for month 5...
  Computing threshold for month 6...
  Computing threshold for month 7...
  Computing threshold for month 8...
  Computing threshold for month 9...
  Computing threshold for month 10...
  Computing threshold for month 11...
  Computing threshold for month 12...
✓ Completed computing thresholds for all 12 months

Computing TX90p for month 1...
Computing TX90p for month 2...
Computing TX90p for month 3...
Computing TX90p for month 4...
Computing TX90p for month 5...
Computing TX90p for month 6...
Computing TX90p for month 7...
Computing TX90p for month 8...
Computing TX90p for month 9...
Computing TX90p for month 10...
Computing TX90p for month 11...
Computing TX90p for month 12...
✓ Completed computing TX90p for all 12 months


In [35]:
# Add metadata and export TX90p monthly indices
# Note: TX90p images have 2 bands: TX90p_pct and TX90p_days
for month in range(1, 13):
    month_name = month_names[month]
    
    # Add metadata to climatology (multi-band: TX90p_pct, TX90p_days)
    tx90p_monthly_clim[month] = with_common_metadata(
        tx90p_monthly_clim[month],
        units="% and days",
        metric="TX90p",
        roi_name="Porto Alegre, Brazil",
        source="ECMWF/ERA5_LAND/DAILY_AGGR",
        extra={
            "type": "climatology",
            "baseline_start_year": 1995,
            "baseline_end_year": 2014,
            "statistic": "mean",
            "month": month,
            "bands": ["TX90p_pct", "TX90p_days"],
            "description": (
                f"Climatological mean of monthly percentage of days when Tmax > 90th percentile "
                f"for {month_name} computed from TX90p across 1995–2014 baseline. "
                f"Bands: TX90p_pct (percentage) and TX90p_days (number of days)."
            )
        }
    )
    
    # Add metadata to 2024 value (multi-band: TX90p_pct, TX90p_days)
    tx90p_monthly_2024[month] = with_common_metadata(
        tx90p_monthly_2024[month],
        units="% and days",
        metric="TX90p",
        roi_name="Porto Alegre, Brazil",
        source="ECMWF/ERA5_LAND/DAILY_AGGR",
        extra={
            "type": "single_month",
            "year": 2024,
            "month": month,
            "bands": ["TX90p_pct", "TX90p_days"],
            "description": (
                f"Monthly percentage of days when Tmax > 90th percentile for {month_name} 2024. "
                f"Bands: TX90p_pct (percentage) and TX90p_days (number of days)."
            )
        }
    )
    
    # Export climatology (both bands together)
    task = ee.batch.Export.image.toDrive(
        image=tx90p_monthly_clim[month],
        description=f"TX90p_Climatology_{month_name}",
        folder="GEE_exports",
        fileNamePrefix=f"TX90p_climatology_{month_name}",
        region=roi_ee,
        scale=11132,
        crs="EPSG:4326",
        maxPixels=1e13
    )
    task.start()
    
    # Export 2024 value (both bands together)
    task = ee.batch.Export.image.toDrive(
        image=tx90p_monthly_2024[month],
        description=f"TX90p_2024_{month_name}",
        folder="GEE_exports",
        fileNamePrefix=f"TX90p_2024_{month_name}",
        region=roi_ee,
        scale=11132,
        crs="EPSG:4326",
        maxPixels=1e13
    )
    task.start()
    
    print(f"✓ Exported TX90p for {month_name} (2 bands: TX90p_pct, TX90p_days)")

print("\n✓ Completed exporting all TX90p monthly indices")

✓ Exported TX90p for Jan (2 bands: TX90p_pct, TX90p_days)
✓ Exported TX90p for Feb (2 bands: TX90p_pct, TX90p_days)
✓ Exported TX90p for Mar (2 bands: TX90p_pct, TX90p_days)
✓ Exported TX90p for Apr (2 bands: TX90p_pct, TX90p_days)
✓ Exported TX90p for May (2 bands: TX90p_pct, TX90p_days)
✓ Exported TX90p for Jun (2 bands: TX90p_pct, TX90p_days)
✓ Exported TX90p for Jul (2 bands: TX90p_pct, TX90p_days)
✓ Exported TX90p for Aug (2 bands: TX90p_pct, TX90p_days)
✓ Exported TX90p for Sep (2 bands: TX90p_pct, TX90p_days)
✓ Exported TX90p for Oct (2 bands: TX90p_pct, TX90p_days)
✓ Exported TX90p for Nov (2 bands: TX90p_pct, TX90p_days)
✓ Exported TX90p for Dec (2 bands: TX90p_pct, TX90p_days)

✓ Completed exporting all TX90p monthly indices


In [36]:
# Loop through all 12 months to compute TX99p climatology and 2024 values
# Step 1: Compute monthly thresholds for all months
tx99_thr_monthly = {}
print("Computing monthly thresholds for TX99p...")
for month in range(1, 13):
    print(f"  Computing threshold for month {month}...")
    tx99_thr_monthly[month] = txx_threshold_monthly(
        tmax_base, month=month, percentile=99, band_name='temperature_2m_max'
    )
print("✓ Completed computing thresholds for all 12 months\n")

# Step 2: Compute TX99p climatology and 2024 values for all months
tx99p_monthly_clim = {}
tx99p_monthly_2024 = {}

for month in range(1, 13):
    print(f"Computing TX99p for month {month}...")
    
    # Climatology (mean TX99p for this month across 1995-2014)
    tx99p_monthly_clim[month] = txxp_monthly_climatology(
        tmax_base, BASE_START_YEAR, BASE_END_YEAR, month=month,
        thr_img=tx99_thr_monthly[month], band_name='temperature_2m_max', roi=roi_ee,
        percentile=99, reducer=ee.Reducer.mean()
    )
    
    # TX99p for this month in 2024
    tx99p_monthly_2024[month] = txxp_for_month(
        tmax, 2024, month=month, thr_img=tx99_thr_monthly[month],
        band_name='temperature_2m_max', roi=roi_ee, percentile=99
    )

print("✓ Completed computing TX99p for all 12 months")

# Access results like:
# tx99p_monthly_clim[1]  # January climatology (has bands: TX99p_pct, TX99p_days)
# tx99p_monthly_2024[1]  # January 2024 (has bands: TX99p_pct, TX99p_days)
# etc.


Computing monthly thresholds for TX99p...
  Computing threshold for month 1...
  Computing threshold for month 2...
  Computing threshold for month 3...
  Computing threshold for month 4...
  Computing threshold for month 5...
  Computing threshold for month 6...
  Computing threshold for month 7...
  Computing threshold for month 8...
  Computing threshold for month 9...
  Computing threshold for month 10...
  Computing threshold for month 11...
  Computing threshold for month 12...
✓ Completed computing thresholds for all 12 months

Computing TX99p for month 1...
Computing TX99p for month 2...
Computing TX99p for month 3...
Computing TX99p for month 4...
Computing TX99p for month 5...
Computing TX99p for month 6...
Computing TX99p for month 7...
Computing TX99p for month 8...
Computing TX99p for month 9...
Computing TX99p for month 10...
Computing TX99p for month 11...
Computing TX99p for month 12...
✓ Completed computing TX99p for all 12 months


In [37]:
# Add metadata and export TX99p monthly indices
# Note: TX99p images have 2 bands: TX99p_pct and TX99p_days
for month in range(1, 13):
    month_name = month_names[month]
    
    # Add metadata to climatology
    tx99p_monthly_clim[month] = with_common_metadata(
        tx99p_monthly_clim[month],
        units="% and days",
        metric="TX99p",
        roi_name="Porto Alegre, Brazil",
        source="ECMWF/ERA5_LAND/DAILY_AGGR",
        extra={
            "type": "climatology",
            "baseline_start_year": 1995,
            "baseline_end_year": 2014,
            "statistic": "mean",
            "month": month,
            "bands": ["TX99p_pct", "TX99p_days"],
            "description": (
                f"Climatological mean of monthly percentage of days when Tmax > 99th percentile "
                f"for {month_name} computed from TX99p across 1995–2014 baseline."
            )
        }
    )
    
    # Add metadata to 2024 value
    tx99p_monthly_2024[month] = with_common_metadata(
        tx99p_monthly_2024[month],
        units="% and days",
        metric="TX99p",
        roi_name="Porto Alegre, Brazil",
        source="ECMWF/ERA5_LAND/DAILY_AGGR",
        extra={
            "type": "single_month",
            "year": 2024,
            "month": month,
            "bands": ["TX99p_pct", "TX99p_days"],
            "description": f"Monthly percentage of days when Tmax > 99th percentile for {month_name} 2024"
        }
    )
    
    # Export climatology
    task = ee.batch.Export.image.toDrive(
        image=tx99p_monthly_clim[month],
        description=f"TX99p_Climatology_{month_name}",
        folder="GEE_exports",
        fileNamePrefix=f"TX99p_climatology_{month_name}",
        region=roi_ee,
        scale=11132,
        crs="EPSG:4326",
        maxPixels=1e13
    )
    task.start()
    
    # Export 2024 value
    task = ee.batch.Export.image.toDrive(
        image=tx99p_monthly_2024[month],
        description=f"TX99p_2024_{month_name}",
        folder="GEE_exports",
        fileNamePrefix=f"TX99p_2024_{month_name}",
        region=roi_ee,
        scale=11132,
        crs="EPSG:4326",
        maxPixels=1e13
    )
    task.start()
    
    print(f"✓ Exported TX99p for {month_name} (2 bands: TX99p_pct, TX99p_days)")

print("\n✓ Completed exporting all TX99p monthly indices")

✓ Exported TX99p for Jan (2 bands: TX99p_pct, TX99p_days)
✓ Exported TX99p for Feb (2 bands: TX99p_pct, TX99p_days)
✓ Exported TX99p for Mar (2 bands: TX99p_pct, TX99p_days)
✓ Exported TX99p for Apr (2 bands: TX99p_pct, TX99p_days)
✓ Exported TX99p for May (2 bands: TX99p_pct, TX99p_days)
✓ Exported TX99p for Jun (2 bands: TX99p_pct, TX99p_days)
✓ Exported TX99p for Jul (2 bands: TX99p_pct, TX99p_days)
✓ Exported TX99p for Aug (2 bands: TX99p_pct, TX99p_days)
✓ Exported TX99p for Sep (2 bands: TX99p_pct, TX99p_days)
✓ Exported TX99p for Oct (2 bands: TX99p_pct, TX99p_days)
✓ Exported TX99p for Nov (2 bands: TX99p_pct, TX99p_days)
✓ Exported TX99p for Dec (2 bands: TX99p_pct, TX99p_days)

✓ Completed exporting all TX99p monthly indices


# HWD

HWD: heat waves duration

### For a specific year

In [14]:
def compute_tx90_baseline_image(tmax_base_ic):
    """
    Compute baseline 90th percentile of daily Tmax (°C)
    over 1995–2014, per pixel.
    """
    def to_celsius(img):
        # K -> °C
        return img.subtract(273.15).rename('tmax')

    base_c = tmax_base_ic.map(to_celsius)

    # Percentile over time (ImageCollection → Image)
    percs = base_c.reduce(ee.Reducer.percentile([90]))
    tx90 = percs.select('tmax_p90').rename('TX90')

    # Optional: clip to ROI for smaller footprint
    return tx90.clip(roi_ee)

In [15]:
tx90_img = compute_tx90_baseline_image(tmax_base)

In [38]:
def make_flags_list(tmax_ic, tx90_img):
    """
    From an ImageCollection of daily Tmax in K, build a List of daily 0/1 flag Images:
    flag = 1 if Tmax(°C) > TX90, else 0.
    Returns a list with an extra sentinel 0-image at the end.
    """
    def to_flag(img):
        t_c = img.subtract(273.15).rename('tmax')  # K → °C
        hot = t_c.gt(tx90_img)
        flag01 = hot.where(hot, 1).where(hot.Not(), 0)
        return flag01.rename('flag').copyProperties(img, ['system:time_start'])

    flags_ic = tmax_ic.map(to_flag).sort('system:time_start')
    n = flags_ic.size()
    flags_list = flags_ic.toList(n)

    # Sentinel 0 at the end to close final run
    first_flag = ee.Image(flags_list.get(0))
    zero_flag = first_flag.multiply(0).rename('flag')
    flags_list_ext = flags_list.cat([zero_flag])

    return flags_list_ext

In [39]:
def compute_hwd_for_year(tmax_ic, tx90_img, min_length=3):
    """
    HWD: total number of heatwave days (days in runs of length >= min_length).
    Returns an ee.Image with band 'HWD'.
    """
    flags_list_ext = make_flags_list(tmax_ic, tx90_img)

    # Initial state: [run, hwd] = [0, 0]
    init_state = ee.Image.constant([0, 0]).rename(['run', 'hwd']).toFloat()

    def step(img, state_img):
        state_img = ee.Image(state_img)
        flag = ee.Image(img).select('flag')

        run = state_img.select('run')
        hwd = state_img.select('hwd')

        # Today's run length
        run_today = run.add(flag).where(flag.eq(0), 0)

        # Runs ending today (flag goes to 0)
        ended = flag.eq(0)

        # Add previous run length to HWD if >= min_length
        valid_run = run.where(run.gte(min_length), run) \
                       .where(run.lt(min_length), 0)
        hwd_new = hwd.add(valid_run.updateMask(ended))

        new_state = ee.Image.cat([
            run_today.rename('run'),
            hwd_new.rename('hwd')
        ]).toFloat()

        return new_state

    final_state = ee.Image(flags_list_ext.iterate(step, init_state))
    hwd_img = final_state.select('hwd').rename('HWD')
    return hwd_img.clip(roi_ee)

ee.Image with one band HWD where each pixel is the number of heatwave days in 2024

In [41]:
# HWD per pixel for 2024 (L = 3 days)
hwd_2024 = compute_hwd_for_year(
    tmax_ic=tmax,
    tx90_img=tx90_img,   # your baseline 90th percentile image from 1995–2014
    min_length=3
)

In [47]:
def compute_hwd_climatology(
    tmax_base_ic,
    tx90_img,
    min_length=3,
    start_year=BASE_START_YEAR,
    end_year=BASE_END_YEAR
):
    """
    Climatological mean HWD over [start_year, end_year], per pixel.
    Output band: 'HWD_clim'
    """
    years = ee.List.sequence(start_year, end_year)

    def per_year(y):
        y = ee.Number(y).int()
        year_ic = tmax_base_ic.filter(
            ee.Filter.calendarRange(y, y, 'year')
        )
        hwd_y = compute_hwd_for_year(year_ic, tx90_img, min_length)
        return hwd_y.set('year', y)

    hwd_yearly_ic = ee.ImageCollection(years.map(per_year))

    hwd_clim = hwd_yearly_ic.mean().rename('HWD_clim').clip(roi_ee)
    return hwd_clim


In [48]:
HWD_clim = compute_hwd_climatology(
    tmax_base, 
    tx90_img, 
    min_length=3
    )

In [53]:
task = ee.batch.Export.image.toDrive(
    image=HWD_clim,
    description="HWD_Climatology",
    folder="GEE_exports",
    fileNamePrefix="HWD_climatology_pot",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

task = ee.batch.Export.image.toDrive(
    image=hwd_2024,
    description="HWD_2024",
    folder="GEE_exports",
    fileNamePrefix="HWD_2024_pot",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

In [43]:
def compute_hwm_for_year(tmax_ic, tx90_img, min_length=3):
    """
    HWM: maximum duration (in days) of any heatwave run in the year.
    Returns an ee.Image with band 'HWM'.
    """
    flags_list_ext = make_flags_list(tmax_ic, tx90_img)

    init_state = ee.Image.constant([0, 0]).rename(['run', 'hwm']).toFloat()

    def step(img, state_img):
        state_img = ee.Image(state_img)
        flag = ee.Image(img).select('flag')

        run = state_img.select('run')
        hwm = state_img.select('hwm')

        # Today's run
        run_today = run.add(flag).where(flag.eq(0), 0)

        # Update HWM
        hwm_new = hwm.max(run_today)

        new_state = ee.Image.cat([
            run_today.rename('run'),
            hwm_new.rename('hwm')
        ]).toFloat()

        return new_state

    final_state = ee.Image(flags_list_ext.iterate(step, init_state))
    hwm_img = final_state.select('hwm').rename('HWM')
    return hwm_img.clip(roi_ee)

In [44]:
hwm_2024 = compute_hwm_for_year(
    tmax, 
    tx90_img, 
    min_length=3
    )

In [49]:
def compute_hwm_climatology(
    tmax_base_ic,
    tx90_img,
    min_length=3,
    start_year=BASE_START_YEAR,
    end_year=BASE_END_YEAR
):
    """
    Climatological mean HWM over [start_year, end_year], per pixel.
    Output band: 'HWM_clim'
    """
    years = ee.List.sequence(start_year, end_year)

    def per_year(y):
        y = ee.Number(y).int()
        year_ic = tmax_base_ic.filter(
            ee.Filter.calendarRange(y, y, 'year')
        )
        hwm_y = compute_hwm_for_year(year_ic, tx90_img, min_length)
        return hwm_y.set('year', y)

    hwm_yearly_ic = ee.ImageCollection(years.map(per_year))

    hwm_clim = hwm_yearly_ic.mean().rename('HWM_clim').clip(roi_ee)
    return hwm_clim

In [50]:
HWM_clim = compute_hwm_climatology(
    tmax_base, 
    tx90_img, 
    min_length=3
    )

In [54]:
task = ee.batch.Export.image.toDrive(
    image=HWM_clim,
    description="HWM_Climatology",
    folder="GEE_exports",
    fileNamePrefix="HWM_climatology_pot",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

task = ee.batch.Export.image.toDrive(
    image=hwm_2024,
    description="HWM_2024",
    folder="GEE_exports",
    fileNamePrefix="HWM_2024_pot",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

In [45]:
def compute_hwn_for_year(tmax_ic, tx90_img, min_length=3):
    """
    HWN: number of heatwave events (runs with length >= min_length).
    Returns an ee.Image with band 'HWN'.
    """
    flags_list_ext = make_flags_list(tmax_ic, tx90_img)

    init_state = ee.Image.constant([0, 0]).rename(['run', 'hwn']).toFloat()

    def step(img, state_img):
        state_img = ee.Image(state_img)
        flag = ee.Image(img).select('flag')

        run = state_img.select('run')
        hwn = state_img.select('hwn')

        # Today's run
        run_today = run.add(flag).where(flag.eq(0), 0)

        # Runs ending today
        ended = flag.eq(0)

        # Event if previous run >= min_length
        event_flag = run.gte(min_length).updateMask(ended)
        hwn_new = hwn.add(event_flag.where(event_flag, 1).where(event_flag.Not(), 0))

        new_state = ee.Image.cat([
            run_today.rename('run'),
            hwn_new.rename('hwn')
        ]).toFloat()

        return new_state

    final_state = ee.Image(flags_list_ext.iterate(step, init_state))
    hwn_img = final_state.select('hwn').rename('HWN')
    return hwn_img.clip(roi_ee)

In [46]:
hwn_2024 = compute_hwn_for_year(
    tmax, 
    tx90_img, 
    min_length=3
    )

In [ ]:
def compute_hwn_climatology(
    tmax_base_ic,
    tx90_img,
    min_length=3,
    start_year=BASE_START_YEAR,
    end_year=BASE_END_YEAR
):
    """
    Climatological mean HWN over [start_year, end_year], per pixel.
    Output band: 'HWN_clim'
    """
    years = ee.List.sequence(start_year, end_year)

    def per_year(y):
        y = ee.Number(y).int()
        year_ic = tmax_base_ic.filter(
            ee.Filter.calendarRange(y, y, 'year')
        )
        hwn_y = compute_hwn_for_year(year_ic, tx90_img, min_length)
        return hwn_y.set('year', y)

    hwn_yearly_ic = ee.ImageCollection(years.map(per_year))

    hwn_clim = hwn_yearly_ic.mean().rename('HWN_clim').clip(roi_ee)
    return hwn_clim

In [52]:
HWN_clim = compute_hwn_climatology(
    tmax_base, 
    tx90_img, 
    min_length=3
    )

In [55]:
task = ee.batch.Export.image.toDrive(
    image=HWN_clim,
    description="HWN_Climatology",
    folder="GEE_exports",
    fileNamePrefix="HWN_climatology_pot",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

task = ee.batch.Export.image.toDrive(
    image=hwm_2024,
    description="HWN_2024",
    folder="GEE_exports",
    fileNamePrefix="HWN_2024_pot",
    region=roi_ee,
    scale=11132,          # ERA5 LAND native resolution
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

## VIZ